# QRT Asset Allocation - ensemble equal3

Predire si le rendement futur de chaque allocation est positif (`prediction = 1[target > 0]`),
metrique : accuracy. Modele : moyenne des probabilites de trois membres (logistique,
ridge sur `asinh(target)`, LightGBM) ajustes sur le meme design, seuil 0.5.

Regles suivies partout :
- un `TS` n'est jamais coupe entre train et validation (les allocations d'un batch partagent un facteur commun) ;
- `TS` ne sert qu'a grouper : pas de variable, pas de chronologie supposee ;
- `ROW_ID` sert uniquement a aligner la soumission (un 1-NN sur `ROW_ID` fait 100 % in-sample : pure memorisation) ;
- toute statistique (imputation, standardisation, one-hot) est apprise sur le train du fold.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from scipy import sparse
from scipy.stats import t as student_t
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", message="X does not have valid feature names")
DATA = Path("data")  # X_train, y_train et X_test du challenge, a cote du notebook

X = pd.read_csv(DATA / "X_train_9xQjqvZ.csv")
train = X.merge(pd.read_csv(DATA / "y_train_Ppwhaz8.csv"), on="ROW_ID", validate="one_to_one")
train["label"] = (train["target"] > 0).astype(np.int8)
test = pd.read_csv(DATA / "X_test_1zTtEnD.csv")

assert not set(train["TS"]) & set(test["TS"]), "aucun TS commun train/test"
assert not train.duplicated(["TS", "ALLOCATION"]).any()
print(f"train {train.shape}, {train['TS'].nunique()} TS | test {test.shape}, {test['TS'].nunique()} TS | "
      f"taux positif {train['label'].mean():.4f}")

train (527073, 47), 2522 TS | test (31870, 45), 120 TS | taux positif 0.5072


## Design fold-local

Variables brutes imputees par la mediane (+ indicateurs de manquants) et standardisees,
one-hot `GROUP`/`ALLOCATION`, et 18 variables **cross-sectionnelles** : pour 5 sources
(`RET_1`, moyenne des 3 derniers rendements, volatilite, taux de volumes manquants,
`log1p(turnover)`), le rang, l'ecart a la mediane et le z-score MAD **dans le TS courant**,
plus 3 interactions. Elles supposent que tout le batch est observe avant de predire.

In [2]:
RET = [f"RET_{i}" for i in range(1, 21)]
VOL = [f"SIGNED_VOLUME_{i}" for i in range(1, 21)]
NUM = RET + VOL + ["MEDIAN_DAILY_TURNOVER"]


def sources(x, ret_fill, turn_fill):
    r = x[RET].to_numpy(float)
    r = np.where(np.isfinite(r), r, ret_fill)
    turn = x["MEDIAN_DAILY_TURNOVER"].to_numpy(float)
    turn = np.where(np.isfinite(turn), turn, turn_fill)
    return {"ret1": r[:, 0], "short": r[:, :3].mean(1), "vol": r.std(1),
            "miss": np.mean(~np.isfinite(x[VOL].to_numpy(float)), axis=1), "turn": np.log1p(np.maximum(turn, 0))}


def cross_section(x, fit):
    ts = x["TS"].to_numpy()
    cols, z, rank = [], {}, {}
    for name, v in sources(x, fit["ret_fill"], fit["turn_fill"]).items():
        g = pd.Series(v).groupby(ts)
        n = g.transform("size").to_numpy(float)
        rank[name] = np.divide(g.rank().to_numpy() - 1, n - 1, out=np.full(len(v), 0.5), where=n > 1) - 0.5
        delta = v - g.transform("median").to_numpy()
        mad = pd.Series(np.abs(delta)).groupby(ts).transform("median").to_numpy()
        z[name] = np.clip(delta / np.maximum(1.4826 * mad, fit["floor"][name]), -8, 8)
        cols += [rank[name], delta, z[name]]
    miss = sources(x, fit["ret_fill"], fit["turn_fill"])["miss"]
    cols += [z["ret1"] * miss, z["short"] * miss, rank["short"] * miss]
    return np.nan_to_num(np.column_stack(cols), nan=0.0, posinf=8.0, neginf=-8.0)


class Design:
    # Tout est appris dans fit (train du fold) puis seulement applique dans transform.
    def fit(self, x):
        ret_fill = np.nanmedian(x[RET].to_numpy(float), axis=0)
        turn_fill = float(np.nanmedian(x["MEDIAN_DAILY_TURNOVER"].to_numpy(float)))
        self.cs = {"ret_fill": np.where(np.isfinite(ret_fill), ret_fill, 0.0),
                   "turn_fill": turn_fill if np.isfinite(turn_fill) else 0.0}
        self.cs["floor"] = {k: max(1.4826 * np.median(np.abs(v - np.median(v))) * 1e-3, 1e-12)
                            for k, v in sources(x, self.cs["ret_fill"], self.cs["turn_fill"]).items()}
        self.imp = SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True).fit(x[NUM])
        self.num_scaler = StandardScaler().fit(self.imp.transform(x[NUM]))
        self.cs_scaler = StandardScaler().fit(cross_section(x, self.cs))
        self.ohe = OneHotEncoder(handle_unknown="ignore").fit(x[["GROUP", "ALLOCATION"]])
        return self

    def transform(self, x):
        num = np.hstack([self.num_scaler.transform(self.imp.transform(x[NUM])),
                         self.cs_scaler.transform(cross_section(x, self.cs))])
        cat = self.ohe.transform(x[["GROUP", "ALLOCATION"]])
        # Produit par l'identite : garde la disposition creuse d'origine (le solveur lsqr y est sensible).
        cat = (cat @ sparse.diags(np.ones(cat.shape[1]), format="csr")).tocsr()
        return sparse.hstack([sparse.csr_matrix(num), cat], format="csr")

## Les trois membres

- **logistique ridge** (`C = 1`) sur le signe ;
- **ridge `asinh`** : on regresse `z = asinh(target / s)`, `s = mediane|target|`. `asinh` comprime les
  queues mais garde le signe, donc la frontiere de decision. `P(target > 0) = F_t5(z_hat / sigma)` ;
- **LightGBM** a hyperparametres figes (jamais regles), peu correle aux deux modeles lineaires.

In [3]:
LOGIT = dict(C=1.0, solver="newton-cholesky", max_iter=100, tol=1e-7, random_state=0)
LGBM = dict(objective="binary", n_estimators=250, learning_rate=0.04, num_leaves=15, min_child_samples=500,
            reg_alpha=0.25, reg_lambda=5.0, subsample=1.0, colsample_bytree=1.0, random_state=0,
            n_jobs=4, deterministic=True, force_col_wise=True, verbosity=-1)
MEMBERS = ("logit", "asinh", "lgbm")


def fit_predict(tr, ev):
    design = Design().fit(tr)
    A, B = design.transform(tr), design.transform(ev)
    y, target = tr["label"].to_numpy(np.int8), tr["target"].to_numpy(float)
    p = {"logit": LogisticRegression(**LOGIT).fit(A, y).predict_proba(B)[:, 1]}
    z = np.arcsinh(target / max(np.median(np.abs(target)), 1e-6))
    ridge = Ridge(alpha=1.0, solver="lsqr", tol=1e-6, max_iter=2000).fit(A, z)
    sigma = max(np.sqrt(np.mean((z - ridge.predict(A)) ** 2)) * np.sqrt(3 / 5), 0.05)
    p["asinh"] = np.clip(student_t.cdf(ridge.predict(B) / sigma, df=5.0), 1e-7, 1 - 1e-7)
    p["lgbm"] = LGBMClassifier(**LGBM).fit(A, y).predict_proba(B)[:, 1]
    return p


def bagged(tr, ev, seeds=(2711, 2729, 2741)):
    # Chaque membre est ajuste sur 3 sous-echantillons de 80 % des TS, scores moyennes.
    ts = np.array(sorted(tr["TS"].astype(str).unique()))
    runs = []
    for seed in seeds:
        keep = np.random.default_rng(seed).choice(ts, size=int(round(0.8 * len(ts))), replace=False)
        runs.append(fit_predict(tr[tr["TS"].isin(keep)], ev))
    return {k: np.mean([r[k] for r in runs], axis=0) for k in MEMBERS}

## Validation : 5 folds groupes par TS

Une repetition, sans bagging (environ 3 minutes). L'incertitude du delta est un bootstrap
qui reechantillonne des **TS** entiers, puisque les lignes d'un meme batch ne sont pas independantes.

In [4]:
folds = np.array_split(np.random.default_rng(2711).permutation(np.array(sorted(train["TS"].unique()))), 5)
oof = {k: np.zeros(len(train)) for k in MEMBERS}
for val_ts in folds:
    va = train["TS"].isin(val_ts).to_numpy()
    assert not set(train.loc[~va, "TS"]) & set(val_ts)
    for k, v in fit_predict(train[~va], train[va]).items():
        oof[k][va] = v
oof["equal3"] = np.mean([oof[k] for k in MEMBERS], axis=0)


def ts_bootstrap_ci(correct, reference, ts, n=2000, seed=48637):
    codes, _ = pd.factorize(ts)
    rows = np.bincount(codes)
    diff = np.bincount(codes, weights=correct.astype(float) - reference)
    w = np.random.default_rng(seed).multinomial(len(rows), np.full(len(rows), 1 / len(rows)), size=n)
    return np.quantile((w @ diff) / (w @ rows), [0.025, 0.975])


y = train["label"].to_numpy()
correct = {k: (v >= 0.5) == y for k, v in oof.items()}
table = pd.DataFrame({k: {"accuracy": c.mean(), "delta vs logit": c.mean() - correct["logit"].mean()}
                      for k, c in correct.items()}).T
table[["IC TS bas", "IC TS haut"]] = [ts_bootstrap_ci(c, correct["logit"], train["TS"]) for c in correct.values()]
table.round(5)

,accuracy,delta vs logit,IC TS bas,IC TS haut
logit,0.52490,0.00000,0.00000,0.00000
asinh,0.52488,-0.00002,-0.00103,0.00104
lgbm,0.52531,0.00041,-0.00146,0.00231
equal3,0.52651,0.00161,0.00064,0.00258


Sur le **holdout verrouille** du 2026-09-05 (504 TS jamais vus, protocole fixe avant le test),
equal3 faisait 0.5244 contre 0.5232 pour la logistique, 0.5225 pour `asinh` et 0.5234 pour LightGBM.
Il bat ses trois membres et le test t par TS etait significatif (p = 0.006), mais l'IC bootstrap du gain
([-0.0010 ; +0.0034]) couvre zero et le gain venait surtout des petits batches.
C'est donc un **candidat retenu avec evidence mixte**, pas un modele prouve superieur.

## Soumission

In [5]:
scores = bagged(train, test)
final = np.mean([scores[k] for k in MEMBERS], axis=0)
submission = pd.DataFrame({"ROW_ID": test["ROW_ID"], "prediction": (final >= 0.5).astype(int)})

assert list(submission.columns) == ["ROW_ID", "prediction"] and len(submission) == len(test)
assert submission["ROW_ID"].is_unique and submission["prediction"].isin([0, 1]).all()
submission.to_csv("submission_equal3.csv", index=False)
print(f"{len(submission)} lignes, taux de predictions positives {submission['prediction'].mean():.6f}")

31870 lignes, taux de predictions positives 0.574898


## Limites

- Les ecarts entre modeles (~0.1-0.2 point) sont bien plus petits que le bruit du classement (~+/- 1.4 point).
- Le choix des trois membres a ete fait apres coup ; le holdout est maintenant consomme.
- Les variables cross-sectionnelles exigent que tout le batch soit disponible au moment de predire.